# 01 - Collect
Clones the framework, authenticates as the notebook identity, and runs every collector. Raw JSON is written to the attached Lakehouse under `Files/fabric-arch-review/<RUN_ID>/raw/`.

This notebook is **deployed and orchestrated by `setup.ipynb`** - you normally do not run it by hand.

## Parameters

In [ ]:
# Parameters (injected by the pipeline)
TENANT_ID = ""
GITHUB_REPO_URL = "https://github.com/biro98/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
WORKSPACE_IDS = ""
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
RUN_ID = "latest"
CAPACITY_METRICS_APP_INSTALLED = False
VERTIPAQ_STATS_READ_DATA = False
ACTIVITY_DAYS_LOG = 7
# Optional service principal. Created/held in a cloud connection at setup;
# Key Vault keys are an alternate opt-in path for unattended scheduled runs
# to run as the notebook identity (default). When set, the secret is read from
# Key Vault — paste it AFTER setup; nothing secret is stored in the notebook.
SP_CLIENT_ID = ""
SP_CONNECTION_NAME = "fabric-arch-review-sp"
SP_SECRET_KEYVAULT = ""
SP_SECRET_NAME = ""

## Clone framework + install dependencies

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/fabric-arch-review-run"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
def _git(args, **kw):
    r = subprocess.run(["git"] + args, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = ((r.stderr or "") + (r.stdout or "")).strip()
        raise RuntimeError("git " + " ".join(args) + " failed (rc=" + str(r.returncode) + "): " + out)
    return r

subprocess.run(["git", "config", "--global", "--add", "safe.directory", REPO_DIR])

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _git(["clone", "--branch", GITHUB_BRANCH, "--depth", "1", _url, REPO_DIR])

# A single-branch clone has no origin/<branch> ref for a *different* branch, so reset
# to FETCH_HEAD (the tip we just fetched). Fall back to a clean clone on any failure.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        _git(["-C", REPO_DIR, "remote", "set-url", "origin", _url])
        _git(["-C", REPO_DIR, "fetch", "--depth", "1", "origin", GITHUB_BRANCH])
        _git(["-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        _fresh_clone()
else:
    _fresh_clone()
del _url
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
if _pip.returncode != 0:
    print("WARNING: pip install exited", _pip.returncode, "- a benign dependency-resolver warning is expected on Fabric; continuing.")
print("Repo + deps ready at", REPO_DIR)

## Authenticate

In [ ]:
from collectors import auth

class FabricTokenProvider:
    # Drop-in replacement for collectors.auth.TokenProvider using the Fabric
    # notebook identity for the Fabric/Power BI/storage/Key Vault planes.
    def __init__(self):
        import notebookutils
        self._get = notebookutils.credentials.getToken

    @staticmethod
    def _audience(scope):
        s = (scope or "").lower()
        if "management.azure.com" in s:
            return "azuremanagement"
        if "vault.azure.net" in s:
            return "keyvault"
        if "storage.azure.com" in s:
            return "storage"
        return "pbi"

    def get_token(self, scope=auth.FABRIC_SCOPE):
        # No caching: notebookutils.getToken returns a fresh, valid token on
        # each call (it manages its own refresh), so long tenant-wide scans
        # never fail on an expired cached token mid-run.
        return self._get(self._audience(scope))

    def headers(self, scope=auth.FABRIC_SCOPE):
        return {"Authorization": "Bearer " + self.get_token(scope), "Content-Type": "application/json"}

# Auth: collectors run as the notebook identity by default. A standing service
# principal is held in the cloud connection SP_CONNECTION_NAME (created at setup);
# to scan unattended as the SPN, schedule this notebook with that SPN as owner.
# Optionally, set SP_CLIENT_ID + SP_SECRET_KEYVAULT + SP_SECRET_NAME to resolve the
# secret from Key Vault for headless scheduled runs.
_use_sp = bool(SP_CLIENT_ID and SP_SECRET_KEYVAULT and SP_SECRET_NAME)
if _use_sp:
    import notebookutils
    _kv = SP_SECRET_KEYVAULT.strip()
    if "vault.azure.net" not in _kv:
        _kv = "https://" + _kv + ".vault.azure.net/"
    os.environ["TENANT_ID"] = TENANT_ID
    os.environ["CLIENT_ID"] = SP_CLIENT_ID
    os.environ["CLIENT_SECRET"] = notebookutils.credentials.getSecret(_kv, SP_SECRET_NAME)
    auth._default_provider = auth.TokenProvider()
    print("Collectors auth ready: service principal", SP_CLIENT_ID)
else:
    auth._default_provider = FabricTokenProvider()
    print("Token acquired:", bool(auth._default_provider.get_token(auth.FABRIC_SCOPE)))
    print("Collectors auth ready: notebook identity")

## Configure + resolve the Lakehouse run folder

In [ ]:
import os

def _flag(v):
    # bool('false') is True in Python, so parse pipeline string params explicitly.
    if isinstance(v, str):
        return "true" if v.strip().lower() in ("1", "true", "yes", "y", "on") else "false"
    return "true" if bool(v) else "false"

if TENANT_ID:
    os.environ["TENANT_ID"] = TENANT_ID
os.environ["CLIENT_NAME"] = CLIENT_NAME
os.environ["ENGAGEMENT_NAME"] = ENGAGEMENT_NAME
os.environ["REVIEWER_NAME"] = REVIEWER_NAME
if WORKSPACE_IDS:
    os.environ["WORKSPACE_IDS"] = WORKSPACE_IDS
os.environ["CAPACITY_METRICS_APP_INSTALLED"] = _flag(CAPACITY_METRICS_APP_INSTALLED)
os.environ["VERTIPAQ_STATS_READ_DATA"] = _flag(VERTIPAQ_STATS_READ_DATA)
os.environ["ACTIVITY_DAYS_LOG"] = str(ACTIVITY_DAYS_LOG).strip() or "7"

LH = "/lakehouse/default/Files"
if not os.path.isdir(LH):
    raise RuntimeError("No default Lakehouse attached to this notebook.")
OUT_DIR = os.path.join(LH, "fabric-arch-review", RUN_ID)
RAW_DIR = os.path.join(OUT_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)
os.environ["OUTPUT_DIR"] = OUT_DIR
print("Run folder:", OUT_DIR)

## Collect (throttle-safe, isolated per collector)

In [ ]:
import importlib, traceback
COLLECTORS = [
    "collectors.tenant_settings",
    "collectors.scanner_api",
    "collectors.workspace_inventory",
    "collectors.git_integration",
    "collectors.capacity_metrics",
    "collectors.capacity_metrics_app",
    "collectors.semantic_models",
    "collectors.semantic_model_definitions",
    "collectors.vertipaq_stats",
    "collectors.best_practices",
    "collectors.lakehouse_warehouse",
    "collectors.pipelines_notebooks",
    "collectors.pipeline_definitions",
    "collectors.deployment_pipelines",
    "collectors.gateways",
    "collectors.realtime_intelligence",
    "collectors.activity_logs",
]
_ok, _failed = 0, []
for name in COLLECTORS:
    print("\n==>", name)
    try:
        importlib.import_module(name).collect(output_dir=RAW_DIR)
        _ok += 1
    except Exception as e:
        _failed.append(name)
        print("  ", name, "failed:", e)
        traceback.print_exc(limit=1)
print("\n=== Collect summary:", str(_ok) + "/" + str(len(COLLECTORS)), "collectors succeeded ===")
if _failed:
    print("Failed collectors:", ", ".join(_failed))
try:
    import notebookutils
    notebookutils.notebook.exit("collected:" + RAW_DIR + " | ok=" + str(_ok) + "/" + str(len(COLLECTORS)))
except Exception:
    pass